# Notebook 03 — MQTT + FastAPI 실시간 파이프라인

**목표:** MQTT 브로커에서 센서 데이터를 수신하여 FastAPI가 실시간으로 이상탐지 결과를 WebSocket으로 전달하는 파이프라인을 구현한다.

**흐름:**
```
MQTT Publisher (센서 시뮬레이터)
        ↓
MQTT Broker (Eclipse Mosquitto)
        ↓
FastAPI (MQTT Subscribe + IsolationForest)
        ↓
WebSocket → 실시간 대시보드
        ↓
SQLite (이상 감지 로그 저장)
```

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import json
import time
import numpy as np
import requests
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

API_BASE = 'http://localhost:8000'
print('라이브러리 로드 완료')
print(f'API 대상: {API_BASE}')

## 1. API 서버 연결 확인

In [ ]:
try:
    r = requests.get(f'{API_BASE}/health', timeout=3)
    print('✅ API 서버 연결 성공')
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f'❌ API 서버 연결 실패: {e}')
    print('uvicorn app.main:app --reload --port 8000 을 먼저 실행하세요')

## 2. 모델 재학습 — POST /model/train

In [ ]:
# data/raw/sensor_data.csv 로 IsolationForest 재학습
r = requests.post(f'{API_BASE}/model/train?contamination=0.05', timeout=30)
result = r.json()
print('=== 학습 결과 ===')
for k, v in result.items():
    print(f'  {k}: {v}')

## 3. REST API 이상탐지 테스트 — POST /detect

In [ ]:
# 정상 / 이상 샘플 10쌍 전송 후 결과 분석
test_cases = [
    # (label, temperature, vibration, current)
    ('정상', 70.0, 0.50, 12.0),
    ('정상', 68.5, 0.48, 11.8),
    ('정상', 72.1, 0.52, 12.3),
    ('과열',  95.0, 0.50, 12.0),
    ('과열', 102.3, 0.49, 11.9),
    ('과진동', 70.0, 2.80, 12.0),
    ('과진동', 69.8, 3.10, 12.1),
    ('전류스파이크', 70.0, 0.50, 28.0),
    ('전류스파이크', 71.0, 0.51, 34.5),
    ('복합이상',  88.0, 1.80, 20.0),
]

results = []
for label, t, v, c in test_cases:
    payload = {'sensor_id': 'nb03_test', 'temperature': t, 'vibration': v, 'current': c}
    r = requests.post(f'{API_BASE}/detect', json=payload, timeout=5)
    d = r.json()
    results.append({
        '유형': label, '온도': t, '진동': v, '전류': c,
        'score': d['anomaly_score'], '이상여부': d['is_anomaly'], '심각도': d['severity']
    })

import pandas as pd
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

## 4. 결과 시각화 — 유형별 이상 점수

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#3B82F6' if not r['이상여부'] else '#EF4444' for r in results]
labels = [r['유형'] for r in results]
scores = [r['score'] for r in results]

# 막대 그래프
ax = axes[0]
bars = ax.barh(labels, scores, color=colors)
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Anomaly Score (낮을수록 이상)')
ax.set_title('유형별 이상 점수')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#3B82F6', label='정상'), Patch(color='#EF4444', label='이상')])

# 심각도 파이차트
ax2 = axes[1]
severity_counts = df_results['심각도'].value_counts()
color_map = {'normal': '#3B82F6', 'warning': '#F59E0B', 'critical': '#EF4444'}
pie_colors = [color_map.get(s, '#9CA3AF') for s in severity_counts.index]
ax2.pie(severity_counts.values, labels=severity_counts.index,
        colors=pie_colors, autopct='%1.0f%%', startangle=90)
ax2.set_title('심각도 분포')

plt.suptitle('REST API 이상탐지 테스트 결과', fontsize=14)
plt.tight_layout()
plt.savefig('../data/api_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. WebSocket 실시간 스트리밍 테스트

In [ ]:
import asyncio
import websockets

WS_URL = 'ws://localhost:8000/ws/stream'
N_MESSAGES = 10

async def test_websocket():
    rng = np.random.RandomState(42)
    ws_results = []

    async with websockets.connect(WS_URL) as ws:
        for i in range(N_MESSAGES):
            # 5개는 정상, 5개는 이상 (과열)
            is_anomaly = i >= 5
            data = {
                'sensor_id': 'ws_test',
                'temperature': float(rng.normal(95, 3) if is_anomaly else rng.normal(70, 3)),
                'vibration':   float(rng.normal(0.5, 0.08)),
                'current':     float(rng.normal(12, 0.8)),
            }
            await ws.send(json.dumps(data))
            response = json.loads(await ws.recv())
            ws_results.append(response)
            status = '🔴 이상' if response['is_anomaly'] else '🟢 정상'
            print(f'[{i+1:02d}] {status} | score={response["anomaly_score"]:.3f} | 온도={response["temperature"]:.1f}°C')

    return ws_results

try:
    ws_results = asyncio.get_event_loop().run_until_complete(test_websocket())
    print(f'\nWebSocket 테스트 완료: {N_MESSAGES}건 처리')
except Exception as e:
    print(f'WebSocket 테스트 실패: {e}')
    print('API 서버가 실행 중인지 확인하세요')

## 6. MQTT 시뮬레이터 개요

MQTT 브로커(Mosquitto)는 `docker-compose up`으로 실행하거나, 로컬 설치 후 직접 구동합니다.

```bash
# 정상 패턴 — 1초마다 publish
python scripts/mqtt_simulator.py

# 이상 패턴 주입
python scripts/mqtt_simulator.py --anomaly
```

### MQTT 메시지 구조
```json
{
  "sensor_id": "sensor_01",
  "temperature": 70.3,
  "vibration": 0.49,
  "current": 12.1,
  "timestamp": "2024-01-15T09:23:11.123"
}
```

### pub/sub 패턴 (산업 현장 적용 시)
```
Topic: sensors/factory/line1
Topic: sensors/factory/line2
Topic: sensors/warehouse/temp
```
→ 계층적 토픽으로 여러 라인/구역을 독립적으로 모니터링 가능

## 7. 통계 API — GET /stats

In [ ]:
r = requests.get(f'{API_BASE}/stats', timeout=5)
stats = r.json()
print('=== 이상 탐지 통계 ===')
print(f'전체 이상 감지: {stats["total_anomalies"]}건')
print(f'오늘 이상 감지: {stats["today_anomalies"]}건')
print(f'Critical: {stats["critical_count"]}건')
print(f'Warning: {stats["warning_count"]}건')
print(f'평균 이상 점수: {stats["avg_anomaly_score"]}')

if stats['top_sensors']:
    print('\n센서별 이상 건수 TOP 5:')
    for s in stats['top_sensors']:
        print(f'  {s["sensor_id"]}: {s["anomaly_count"]}건')

## 8. 파이프라인 요약

| 엔드포인트 | 방식 | 설명 |
|-----------|------|------|
| `POST /detect` | REST | 단건 센서 데이터 이상탐지 |
| `WS /ws/stream` | WebSocket | 실시간 양방향 스트리밍 |
| `POST /model/train` | REST | 새 데이터로 모델 재학습 |
| `GET /logs` | REST | 이상 감지 로그 조회 |
| `GET /stats` | REST | 집계 통계 조회 |
| `GET /health` | REST | 서버 상태 확인 |

**선택 이유:**
- REST는 단건 요청/배치 처리에 적합
- WebSocket은 대시보드처럼 지속 연결이 필요한 클라이언트에 적합
- 두 방식을 동시 제공하여 다양한 클라이언트 요구사항에 대응